# Greedy Optimization for Exercise/Task Assignments

Simple greedy approach that pretty much does the following:
- scores every possible User to each tasker/exercise pairing (priortizing taskers by ordering)
- fills the current highest-priority taskers/exercises first with the best available scoring Users
- enforces hard constraints (via scoring) like: unavailable Users, wrong jobcode/specialty, insufficient rank, etc.

Note using synthetic data based on high-level current state processes with scoring based on what I think might be important (or could be). *Used NIPRGPT (GPT OSS 120B) to create this synthetic User and exercise input data.*

In [1]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

#### User input data 

In [2]:
Users = [
    {
        "id": 1001,
        "jobcode": "0311",
        "rank": "Cpl",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": ["JTAC", "Arabic"],
        "command": "I MEF",
        "past_deployments": 2,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1002,
        "jobcode": "0231",
        "rank": "Sgt",
        "available": False,
        "location": "Quantico",
        "specialties": ["Recon"],
        "command": "II MEF",
        "past_deployments": 3,
        "clearance": "TopSecret",
        "deployment_status": "Deployed",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1003,
        "jobcode": "0211",
        "rank": "LCpl",
        "available": True,
        "location": "Camp Lejeune",
        "specialties": ["Recon"],
        "command": "III MEF",
        "past_deployments": 1,
        "clearance": "TopSecret",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1004,
        "jobcode": "2110",
        "rank": "SgtMaj",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["Cyber", "Signals"],
        "command": "I MEF",
        "past_deployments": 4,
        "clearance": "TopSecret",
        "deployment_status": "Training",
        "last_tasker_type": "IA"
    },
    {
        "id": 1005,
        "jobcode": "0610",
        "rank": "MGySgt",
        "available": False,
        "location": "San Diego",
        "specialties": ["Pilot"],
        "command": "II MEF",
        "past_deployments": 2,
        "clearance": "Secret",
        "deployment_status": "Deployed",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1006,
        "jobcode": "0311",
        "rank": "Pvt",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": [],
        "command": "III MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1007,
        "jobcode": "0231",
        "rank": "SSgt",
        "available": True,
        "location": "Quantico",
        "specialties": ["Medical"],
        "command": "I MEF",
        "past_deployments": 1,
        "clearance": "Secret",
        "deployment_status": "Leave",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1008,
        "jobcode": "0321",
        "rank": "Cpl",
        "available": False,
        "location": "Camp Lejeune",
        "specialties": ["Engineer"],
        "command": "II MEF",
        "past_deployments": 2,
        "clearance": "Confidential",
        "deployment_status": "Medical",
        "last_tasker_type": "IA"
    },
    {
        "id": 1009,
        "jobcode": "2110",
        "rank": "GySgt",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["Signals"],
        "command": "III MEF",
        "past_deployments": 5,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1010,
        "jobcode": "0311",
        "rank": "PFC",
        "available": True,
        "location": "San Diego",
        "specialties": [],
        "command": "I MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1011,
        "jobcode": "0231",
        "rank": "MSgt",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": ["Cyber"],
        "command": "II MEF",
        "past_deployments": 3,
        "clearance": "TopSecret",
        "deployment_status": "None",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1012,
        "jobcode": "0311",
        "rank": "SgtMaj",
        "available": False,
        "location": "Quantico",
        "specialties": ["JTAC"],
        "command": "III MEF",
        "past_deployments": 4,
        "clearance": "Secret",
        "deployment_status": "Deployed",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1013,
        "jobcode": "0321",
        "rank": "Cpl",
        "available": True,
        "location": "Camp Lejeune",
        "specialties": ["Logistics"],
        "command": "I MEF",
        "past_deployments": 1,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1014,
        "jobcode": "2110",
        "rank": "SSgt",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["Engineer"],
        "command": "II MEF",
        "past_deployments": 2,
        "clearance": "Secret",
        "deployment_status": "Training",
        "last_tasker_type": "IA"
    },
    {
        "id": 1015,
        "jobcode": "0610",
        "rank": "SgtMaj",
        "available": False,
        "location": "San Diego",
        "specialties": ["Pilot"],
        "command": "III MEF",
        "past_deployments": 5,
        "clearance": "TopSecret",
        "deployment_status": "Deployed",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1016,
        "jobcode": "0311",
        "rank": "LCpl",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": [],
        "command": "I MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1017,
        "jobcode": "0231",
        "rank": "GySgt",
        "available": True,
        "location": "Quantico",
        "specialties": ["Medical"],
        "command": "II MEF",
        "past_deployments": 1,
        "clearance": "Secret",
        "deployment_status": "Leave",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1018,
        "jobcode": "0321",
        "rank": "MSgt",
        "available": False,
        "location": "Camp Lejeune",
        "specialties": ["Cyber", "Signals"],
        "command": "III MEF",
        "past_deployments": 3,
        "clearance": "TopSecret",
        "deployment_status": "Medical",
        "last_tasker_type": "IA"
    },
    {
        "id": 1019,
        "jobcode": "2110",
        "rank": "Cpl",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["Recon"],
        "command": "I MEF",
        "past_deployments": 2,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1020,
        "jobcode": "0311",
        "rank": "PFC",
        "available": True,
        "location": "San Diego",
        "specialties": [],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1021,
        "jobcode": "0231",
        "rank": "Sgt",
        "available": False,
        "location": "Camp Pendleton",
        "specialties": ["JTAC"],
        "command": "III MEF",
        "past_deployments": 4,
        "clearance": "Secret",
        "deployment_status": "Deployed",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1022,
        "jobcode": "0321",
        "rank": "LCpl",
        "available": True,
        "location": "Quantico",
        "specialties": ["Engineer"],
        "command": "I MEF",
        "past_deployments": 2,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1023,
        "jobcode": "2110",
        "rank": "SgtMaj",
        "available": True,
        "location": "Camp Lejeune",
        "specialties": ["Cyber"],
        "command": "II MEF",
        "past_deployments": 5,
        "clearance": "TopSecret",
        "deployment_status": "Training",
        "last_tasker_type": None
    },
    {
        "id": 1024,
        "jobcode": "0610",
        "rank": "MGySgt",
        "available": False,
        "location": "Twentynine Palms",
        "specialties": ["Pilot"],
        "command": "III MEF",
        "past_deployments": 3,
        "clearance": "Secret",
        "deployment_status": "Deployed",
        "last_tasker_type": "IA"
    },
    {
        "id": 1025,
        "jobcode": "0311",
        "rank": "Sgt",
        "available": True,
        "location": "San Diego",
        "specialties": ["JTAC"],
        "command": "I MEF",
        "past_deployments": 1,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1026,
        "jobcode": "0231",
        "rank": "Pvt",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": [],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1027,
        "jobcode": "0321",
        "rank": "SSgt",
        "available": False,
        "location": "Quantico",
        "specialties": ["Logistics"],
        "command": "III MEF",
        "past_deployments": 2,
        "clearance": "Secret",
        "deployment_status": "Leave",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1028,
        "jobcode": "2110",
        "rank": "Cpl",
        "available": True,
        "location": "Camp Lejeune",
        "specialties": ["Signals"],
        "command": "I MEF",
        "past_deployments": 1,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": "IA"
    },
    {
        "id": 1029,
        "jobcode": "0311",
        "rank": "LCpl",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": [],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1030,
        "jobcode": "0231",
        "rank": "GySgt",
        "available": False,
        "location": "San Diego",
        "specialties": ["Medical"],
        "command": "III MEF",
        "past_deployments": 4,
        "clearance": "Secret",
        "deployment_status": "Medical",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1031,
        "jobcode": "0321",
        "rank": "SgtMaj",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": ["Engineer"],
        "command": "I MEF",
        "past_deployments": 3,
        "clearance": "TopSecret",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1032,
        "jobcode": "2110",
        "rank": "PFC",
        "available": True,
        "location": "Quantico",
        "specialties": [],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1033,
        "jobcode": "0610",
        "rank": "Sgt",
        "available": False,
        "location": "Camp Lejeune",
        "specialties": ["Pilot"],
        "command": "III MEF",
        "past_deployments": 2,
        "clearance": "TopSecret",
        "deployment_status": "Deployed",
        "last_tasker_type": "IA"
    },
    {
        "id": 1034,
        "jobcode": "0311",
        "rank": "Cpl",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["JTAC"],
        "command": "I MEF",
        "past_deployments": 1,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1035,
        "jobcode": "0231",
        "rank": "LCpl",
        "available": True,
        "location": "San Diego",
        "specialties": ["Cyber"],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1036,
        "jobcode": "0321",
        "rank": "Sgt",
        "available": False,
        "location": "Camp Pendleton",
        "specialties": ["Logistics", "Engineer"],
        "command": "III MEF",
        "past_deployments": 3,
        "clearance": "Secret",
        "deployment_status": "Leave",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1037,
        "jobcode": "2110",
        "rank": "SSgt",
        "available": True,
        "location": "Quantico",
        "specialties": ["Signals"],
        "command": "I MEF",
        "past_deployments": 2,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": "IA"
    },
    {
        "id": 1038,
        "jobcode": "0610",
        "rank": "GySgt",
        "available": False,
        "location": "Camp Lejeune",
        "specialties": ["Pilot"],
        "command": "II MEF",
        "past_deployments": 5,
        "clearance": "TopSecret",
        "deployment_status": "Deployed",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1039,
        "jobcode": "0311",
        "rank": "SgtMaj",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["JTAC", "Arabic"],
        "command": "III MEF",
        "past_deployments": 4,
        "clearance": "TopSecret",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1040,
        "jobcode": "0231",
        "rank": "Pvt",
        "available": True,
        "location": "San Diego",
        "specialties": [],
        "command": "I MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1041,
        "jobcode": "0321",
        "rank": "LCpl",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": ["Engineer"],
        "command": "II MEF",
        "past_deployments": 1,
        "clearance": "Secret",
        "deployment_status": "Training",
        "last_tasker_type": "IA"
    },
    {
        "id": 1042,
        "jobcode": "2110",
        "rank": "Cpl",
        "available": False,
        "location": "Quantico",
        "specialties": ["Cyber"],
        "command": "III MEF",
        "past_deployments": 2,
        "clearance": "TopSecret",
        "deployment_status": "Medical",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1043,
        "jobcode": "0610",
        "rank": "Sgt",
        "available": True,
        "location": "Camp Lejeune",
        "specialties": ["Pilot"],
        "command": "I MEF",
        "past_deployments": 3,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1044,
        "jobcode": "0311",
        "rank": "PFC",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": [],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1045,
        "jobcode": "0231",
        "rank": "SgtMaj",
        "available": False,
        "location": "San Diego",
        "specialties": ["Medical"],
        "command": "III MEF",
        "past_deployments": 4,
        "clearance": "TopSecret",
        "deployment_status": "Deployed",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1046,
        "jobcode": "0321",
        "rank": "SSgt",
        "available": True,
        "location": "Camp Pendleton",
        "specialties": ["Logistics"],
        "command": "I MEF",
        "past_deployments": 2,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "JTF"
    },
    {
        "id": 1047,
        "jobcode": "2110",
        "rank": "GySgt",
        "available": True,
        "location": "Quantico",
        "specialties": ["Signals"],
        "command": "II MEF",
        "past_deployments": 1,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    },
    {
        "id": 1048,
        "jobcode": "0610",
        "rank": "MGySgt",
        "available": False,
        "location": "Camp Lejeune",
        "specialties": ["Pilot"],
        "command": "III MEF",
        "past_deployments": 5,
        "clearance": "TopSecret",
        "deployment_status": "Deployed",
        "last_tasker_type": "IA"
    },
    {
        "id": 1049,
        "jobcode": "0311",
        "rank": "Cpl",
        "available": True,
        "location": "Twentynine Palms",
        "specialties": ["JTAC"],
        "command": "I MEF",
        "past_deployments": 2,
        "clearance": "Secret",
        "deployment_status": "None",
        "last_tasker_type": "FAP"
    },
    {
        "id": 1050,
        "jobcode": "0231",
        "rank": "LCpl",
        "available": True,
        "location": "San Diego",
        "specialties": ["Cyber", "Arabic"],
        "command": "II MEF",
        "past_deployments": 0,
        "clearance": "Confidential",
        "deployment_status": "None",
        "last_tasker_type": None
    }
]

#### Exercise input data

In [3]:
exercises = [
    {
        "name": "ExA_FAP",
        "required_count": 2,
        "jobcode_required": "0311",
        "min_rank": "Pvt",
        "grade_required": None,
        "clearance_required": "Secret",
        "specialty_required": None,
        "priority": 1.0,
        "preferred_commands": ["I MEF"],
        "start_date": "2025‑12‑20",
        "due_date": "2025‑12‑27",
    },
    {
        "name": "ExB_JTF",
        "required_count": 1,
        "jobcode_required": "0211",
        "min_rank": "LCpl",
        "grade_required": None,
        "clearance_required": "TopSecret",
        "specialty_required": "Recon",
        "priority": 1.2,
        "preferred_commands": ["II MEF", "III MEF"],
        "start_date": "2025‑12‑22",
        "due_date": "2025‑12‑30",
    },
    {
        "name": "ExC_IA",
        "required_count": 1,
        "jobcode_required": "0231",
        "min_rank": "Sgt",
        "grade_required": None,
        "clearance_required": None,
        "specialty_required": None,
        "priority": 0.8,
        "preferred_commands": [],
        "start_date": "2025‑12‑25",
        "due_date": "2025‑12‑31",
    }
]

In [4]:
rank_order = [
    "Pvt", "PFC", "LCpl", "Cpl", "Sgt", "SSgt", "GySgt",
    "MSgt", "1stSgt", "MGySgt", "SgtMaj"
]

def rank_meets_need(User_rank, required_rank):
    return rank_order.index(User_rank) >= rank_order.index(required_rank)

## Scoring Function

A lot of conditionals, where hard constraints, like the User being available, adjust the score by a large negative values (-9999) to ensure they are not selected. In general, lower (and negative) values indicate less desirable tasker/exercise choices. Simple conditions that do not necessarily reflect reality.

In [5]:
def fitness_fun(User, exercise):
    score = 0

    if not User["available"]:
        return -9999

    if User["jobcode"] == exercise["jobcode_required"]:
        score = score + 30
    else:
        score = score - 50

    # rank/grade requirement only if grade_required is there and not None
    if exercise.get("grade_required") is not None:
        if rank_meets_need(User["rank"], exercise["min_rank"]):
            score =score + 10
        else:
            return -9999

    # clearance requirement only if clearance_required is there and not None
    if exercise.get("clearance_required") is not None:
        if User["clearance"] != exercise["clearance_required"]:
            return -9999

    # specialty requirement only if specialty_required is there and not None
    if exercise.get("specialty_required") is not None:
        if exercise["specialty_required"] in User["specialties"]:
            score = score + 20
        else:
            return -9999

    if User["command"] in exercise["preferred_commands"]:
        score = score + 5

    # deployment penalty if not None / not missing
    # checking string too since that happened...
    if User.get("deployment_status") not in (None, "None"):
        score = score - 5

    # proxy for past deployments fairness (just picked 10 as max possible for now)
    score = score + max(10 - User["past_deployments"], 0)

    score = score * exercise["priority"]

    return score

## Greedy Function

This function basically does the following:
- picks the best first choice (highest score)
- do **not** consider future choices

Local optimum but really hoping for the global one.

This is very similar to selection sort (which no one uses), i.e. find the smallest remaining element and put it in its spot, repeat until done.

In [6]:
def greedy_asssignments(Users, exercises):

    assignments = []
    used = set()

    # sort exercises by priority
    ex_sorted = sorted(exercises, key=lambda ex: ex["priority"], reverse=True)

    for ex in ex_sorted: #loop through each exercise/tasker
        filled = 0
        while filled < ex["required_count"]:
            best_User = None
            best_score = -9999

            for m in Users: #look through each User
                #only 1 assignment per User
                if m["id"] in used:
                    continue

                score = fitness_fun(m, ex)
                if score > best_score:
                    best_score = score
                    best_User = m

            # can't fill
            if best_User is None or best_score < 0:
                break

            assignments.append((best_User["id"], ex["name"]))
            used.add(best_User["id"])
            filled = filled + 1

        ex["slots_filled"] = filled

    return assignments, exercises

### Running and Reporting

In [7]:
def match_key(x, assignments):
    matches = []

    for i in assignments:
        key = i[0]
        value = i[1]
        if value == x:
            matches.append(key)

    return matches

def my_report(assignments, exercises, Users):
    for ex in exercises:
        required = ex["required_count"]
        filled = ex.get("slots_filled", 0)
        status = "MET" if filled == required else "NOT MET"
        m = match_key(ex['name'], assignments)
        print(f"{ex['name']}: required {required}, assigned {filled} ({status} with Users {m})")

In [8]:
assignments, updated_exercises = greedy_asssignments(Users, exercises)

In [9]:
my_report(assignments, updated_exercises, Users)

ExA_FAP: required 2, assigned 2 (MET with Users [1025, 1034])
ExB_JTF: required 1, assigned 1 (MET with Users [1003])
ExC_IA: required 1, assigned 1 (MET with Users [1026])
